# Planning Patterns

**Companion lesson:** https://ml-viz.vercel.app/courses/agent-design-patterns/04-planning-patterns

Implement both planning patterns from scratch — the **Single-path Plan Generator**
and the **Multi-path Plan Generator** — run them on a toy task, score and compare
candidate plans, and visualise the plan tree with matplotlib. No API keys required.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

BG     = '#0f1117'
CARD   = '#1a1d27'
BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
YELLOW = '#fbbf24'
TEXT   = '#e2e8f0'
MUTED  = '#64748b'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': CARD,
    'axes.edgecolor': MUTED, 'text.color': TEXT,
    'axes.labelcolor': TEXT, 'xtick.color': TEXT,
    'ytick.color': TEXT, 'grid.color': MUTED, 'grid.alpha': 0.3,
})
np.random.seed(0)
print('Setup complete.')

## 1  Data structures

In [ ]:
@dataclass
class Goal:
    intent:      str
    entities:    Dict[str, Any] = field(default_factory=dict)
    constraints: Dict[str, Any] = field(default_factory=dict)

@dataclass
class PlanStep:
    action:           str
    params:           Dict[str, Any] = field(default_factory=dict)
    expected_outcome: str = ''
    executed:         bool = False
    result:           Optional[str] = None

@dataclass
class PlanScore:
    feasibility: float
    efficiency:  float
    safety:      float
    weights:     Tuple[float, float, float] = (0.5, 0.3, 0.2)

    @property
    def composite(self) -> float:
        wf, we, ws = self.weights
        return wf * self.feasibility + we * self.efficiency + ws * self.safety

print('Data structures ready.')

## 2  Mock LLM with multiple plan variants

For the multi-path generator we need slightly different plans on each call
to simulate the FM sampling different outputs at temperature > 0.

In [ ]:
# Three distinct plans for the toy goal: "Produce a market research report on EV adoption."
_PLAN_VARIANTS = [
    # Plan A — web-first approach
    json.dumps([
        {"action": "web_search",   "params": {"query": "EV adoption trends 2025"},
         "expected_outcome": "Recent news articles and statistics"},
        {"action": "web_search",   "params": {"query": "EV market share by region"},
         "expected_outcome": "Regional breakdown data"},
        {"action": "summarise",    "params": {"topic": "EV adoption", "max_bullets": 8},
         "expected_outcome": "Structured summary"},
        {"action": "write_report", "params": {"format": "markdown"},
         "expected_outcome": "Final markdown report"}
    ]),
    # Plan B — database-first approach (fewer steps, lower latency)
    json.dumps([
        {"action": "query_database", "params": {"table": "ev_sales", "year": 2025},
         "expected_outcome": "Sales data rows"},
        {"action": "run_analysis",   "params": {"metrics": ["yoy_growth", "market_share"]},
         "expected_outcome": "Computed metrics dict"},
        {"action": "write_report",   "params": {"format": "pdf"},
         "expected_outcome": "Final PDF report"}
    ]),
    # Plan C — hybrid: database + web validation
    json.dumps([
        {"action": "query_database", "params": {"table": "ev_sales", "year": 2025},
         "expected_outcome": "Internal sales data"},
        {"action": "web_search",     "params": {"query": "EV market report 2025"},
         "expected_outcome": "External market data"},
        {"action": "cross_validate", "params": {"sources": ["internal", "web"]},
         "expected_outcome": "Validated data set"},
        {"action": "write_report",   "params": {"format": "markdown"},
         "expected_outcome": "Final validated report"}
    ]),
]

_call_count = [0]

def mock_llm_plan(prompt: str) -> str:
    """Returns a different plan variant each call (round-robin)."""
    idx = _call_count[0] % len(_PLAN_VARIANTS)
    _call_count[0] += 1
    return _PLAN_VARIANTS[idx]

def mock_llm_eval(prompt: str) -> str:
    """Returns a plausible score for each plan variant."""
    # Deterministic scores based on prompt content keywords
    if 'web_search' in prompt and 'cross_validate' in prompt:
        return json.dumps({"feasibility": 8.5, "efficiency": 6.5, "safety": 9.0})  # Plan C
    if 'query_database' in prompt and 'run_analysis' in prompt:
        return json.dumps({"feasibility": 9.0, "efficiency": 9.0, "safety": 7.0})  # Plan B
    return json.dumps({"feasibility": 7.5, "efficiency": 7.0, "safety": 8.0})       # Plan A default

print('Mock LLMs ready. Plan variants:', len(_PLAN_VARIANTS))

## 3  Single-path Plan Generator

In [ ]:
TOOL_CATALOGUE = [
    {"name": "web_search",     "description": "Search the web for information"},
    {"name": "query_database", "description": "Run a SQL-like query on a database"},
    {"name": "run_analysis",   "description": "Apply statistical analysis to data"},
    {"name": "cross_validate", "description": "Cross-validate data from multiple sources"},
    {"name": "summarise",      "description": "Summarise text into bullet points"},
    {"name": "write_report",   "description": "Write a formatted report"},
]

class SinglePathPlanGenerator:
    def __init__(self, llm):
        self.llm = llm

    def generate(self, goal: Goal) -> List[PlanStep]:
        prompt = (
            f"Produce a step-by-step plan to achieve the goal.\n"
            f"Goal: {goal.intent}\n"
            f"Entities: {goal.entities}\n"
            f"Constraints: {goal.constraints}\n"
            f"Available tools: {[t['name'] for t in TOOL_CATALOGUE]}\n"
            f"Return JSON array with action, params, expected_outcome per step.\n"
            f"JSON:"
        )
        raw  = self.llm(prompt)
        data = json.loads(raw)
        return [PlanStep(**s) for s in data]


goal = Goal(
    intent='Produce a market research report on EV adoption',
    entities={'topic': 'EV adoption', 'year': 2025},
    constraints={'format': 'markdown', 'max_pages': 5},
)

_call_count[0] = 0   # reset to get Plan A
single_gen = SinglePathPlanGenerator(llm=mock_llm_plan)
single_plan = single_gen.generate(goal)

print(f'Single-path plan ({len(single_plan)} steps):')
for i, step in enumerate(single_plan, 1):
    print(f'  Step {i}: {step.action}({step.params})')
    print(f'          → {step.expected_outcome}')

## 4  Multi-path Plan Generator with scoring

In [ ]:
class PlanEvaluator:
    def __init__(self, llm):
        self.llm = llm

    def score(self, plan: List[PlanStep], goal: Goal) -> PlanScore:
        plan_text = '\n'.join(f'{i+1}. {s.action}({s.params})' for i, s in enumerate(plan))
        prompt = (
            f"Rate this agent plan for achieving: {goal.intent}\n\n"
            f"Plan:\n{plan_text}\n\n"
            f"Return JSON: {{\"feasibility\": 0-10, \"efficiency\": 0-10, \"safety\": 0-10}}\n"
            f"JSON:"
        )
        raw  = self.llm(prompt)
        data = json.loads(raw)
        return PlanScore(
            feasibility=data['feasibility'],
            efficiency=data['efficiency'],
            safety=data['safety'],
        )


class MultiPathPlanGenerator:
    def __init__(self, plan_llm, eval_llm, n_candidates: int = 3):
        self._single    = SinglePathPlanGenerator(plan_llm)
        self.evaluator  = PlanEvaluator(eval_llm)
        self.n_candidates = n_candidates

    def generate(self, goal: Goal) -> Tuple[List[PlanStep], List[Tuple[List[PlanStep], PlanScore]]]:
        candidates = [self._single.generate(goal) for _ in range(self.n_candidates)]
        scored = [(p, self.evaluator.score(p, goal)) for p in candidates]
        best_plan = max(scored, key=lambda x: x[1].composite)[0]
        return best_plan, scored


_call_count[0] = 0   # reset to cycle through all 3 variants
multi_gen = MultiPathPlanGenerator(plan_llm=mock_llm_plan, eval_llm=mock_llm_eval, n_candidates=3)
best_plan, all_scored = multi_gen.generate(goal)

print('Multi-path candidates and scores:')
for idx, (plan, score) in enumerate(all_scored, 1):
    print(f'\n  Plan {idx} ({len(plan)} steps)  →  '
          f'feasibility={score.feasibility:.1f}  '
          f'efficiency={score.efficiency:.1f}  '
          f'safety={score.safety:.1f}  '
          f'composite={score.composite:.2f}')
    for s in plan:
        print(f'    {s.action}')

best_idx = np.argmax([s.composite for _, s in all_scored])
print(f'\nBest plan: Plan {best_idx + 1}')

## 5  Visualise plan scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: grouped bar chart of dimension scores per plan
ax = axes[0]
labels     = ['Feasibility', 'Efficiency', 'Safety', 'Composite']
bar_colors = [BRAND, TEAL, YELLOW]
n_plans    = len(all_scored)
x          = np.arange(len(labels))
width      = 0.25

for i, (plan, score) in enumerate(all_scored):
    vals = [score.feasibility, score.efficiency, score.safety, score.composite]
    ax.bar(x + (i - 1) * width, vals, width, label=f'Plan {i+1}',
           color=bar_colors[i], alpha=0.85)

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 11); ax.set_ylabel('Score (0–10)')
ax.set_title('Plan scores by dimension', color=TEXT, pad=8)
ax.axhline(7.0, color=MUTED, ls=':', lw=1)
ax.legend()

# Right: plan length (step count) per plan
ax2 = axes[1]
plan_names  = [f'Plan {i+1}' for i in range(n_plans)]
step_counts = [len(p) for p, _ in all_scored]
comp_scores = [s.composite for _, s in all_scored]

scatter = ax2.scatter(step_counts, comp_scores, s=180,
                      c=[BRAND, TEAL, YELLOW], zorder=3)
for i, (sc, cs) in enumerate(zip(step_counts, comp_scores)):
    ax2.annotate(f'Plan {i+1}', (sc, cs), textcoords='offset points',
                 xytext=(8, 4), color=TEXT, fontsize=9)
ax2.set_xlabel('Number of steps (plan length)')
ax2.set_ylabel('Composite score')
ax2.set_title('Efficiency vs. composite score trade-off', color=TEXT, pad=8)
ax2.set_ylim(6, 10)

plt.tight_layout(); plt.show()

## 6  Plan tree visualisation

We can visualise the multi-path structure as a tree: the root is the goal,
branches are alternative plans, leaves are steps.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(-0.5, 7)
ax.axis('off')
ax.set_title('Multi-path plan tree', color=TEXT, pad=10, fontsize=13)

# Draw root (goal)
root_x, root_y = 7, 6.5
ax.add_patch(plt.Circle((root_x, root_y), 0.55, color=BRAND, zorder=3))
ax.text(root_x, root_y, 'Goal', ha='center', va='center',
        color='white', fontsize=9, fontweight='bold', zorder=4)

plan_colors = [BRAND, TEAL, YELLOW]
plan_xs     = [2.5, 7.0, 11.5]
plan_y      = 5.0

for pi, ((plan, score), px, pc) in enumerate(zip(all_scored, plan_xs, plan_colors)):
    # Draw plan node
    ax.annotate('', xy=(px, plan_y + 0.55), xytext=(root_x, root_y - 0.55),
                arrowprops=dict(arrowstyle='->', color=pc, lw=1.5))
    ax.add_patch(plt.Circle((px, plan_y), 0.45, color=pc, zorder=3, alpha=0.9))
    ax.text(px, plan_y, f'P{pi+1}\n{score.composite:.1f}',
            ha='center', va='center', color=BG, fontsize=8, fontweight='bold', zorder=4)

    # Draw step nodes
    n_steps = len(plan)
    step_ys = np.linspace(plan_y - 1.0, plan_y - 1.0 - (n_steps - 1) * 0.85, n_steps)
    for si, (step, sy) in enumerate(zip(plan, step_ys)):
        ax.annotate('', xy=(px, sy + 0.28), xytext=(px, plan_y - 0.45),
                    arrowprops=dict(arrowstyle='->', color=MUTED, lw=1.0))
        ax.add_patch(plt.FancyBboxPatch((px - 1.1, sy - 0.22), 2.2, 0.44,
                                        boxstyle='round,pad=0.05',
                                        fc=CARD, ec=pc, lw=1.2, zorder=3))
        ax.text(px, sy, step.action, ha='center', va='center',
                color=TEXT, fontsize=7.5, zorder=4)

    # Highlight best plan with a star
    if pi == best_idx:
        ax.text(px + 0.6, plan_y + 0.1, '★', color=YELLOW, fontsize=14, zorder=5)

plt.tight_layout(); plt.show()

## ✏️ Your turn

### Exercise A — Plan executor with outcome checking

Implement a `PlanExecutor.run()` that iterates over plan steps, calls
the appropriate mock tool, and returns `'FAILED'` with the failing step
index if any step's result is `None`.

In [ ]:
# Mock tools: all succeed except 'cross_validate' which returns None
MOCK_TOOLS = {
    'web_search':     lambda **kw: f'[WEB] Results for {kw}',
    'query_database': lambda **kw: f'[DB] Rows from {kw}',
    'run_analysis':   lambda **kw: f'[ANALYSIS] Metrics: {kw}',
    'cross_validate': lambda **kw: None,   # always fails
    'summarise':      lambda **kw: f'[SUMMARY] {kw}',
    'write_report':   lambda **kw: f'[REPORT] {kw}',
}

class PlanExecutor:
    def __init__(self, tools: dict):
        self.tools = tools

    def run(self, plan: List[PlanStep]):
        """
        Execute each step in order.
        Returns 'FAILED:<step_index>' if a step result is None,
        or the result of the last step on success.
        """
        # TODO(you): iterate steps, call self.tools[step.action](**step.params),
        # set step.result, return 'FAILED:<i>' if result is None
        pass


# Plan A has no cross_validate, should succeed
plan_a = all_scored[0][0]
executor = PlanExecutor(MOCK_TOOLS)
result_a = executor.run(plan_a)
assert result_a is not None and not str(result_a).startswith('FAILED'), \
    f'Plan A should succeed, got {result_a}'

# Plan C has cross_validate, should fail
plan_c = all_scored[2][0]
result_c = executor.run(plan_c)
assert str(result_c).startswith('FAILED'), f'Plan C should fail, got {result_c}'
print(f'Plan A result: {result_a}')
print(f'Plan C result: {result_c}')
print('passed ✓')

### Exercise B — Custom scoring weights

A safety-critical application should weight safety at 0.6, feasibility at 0.3,
and efficiency at 0.1. Create a `safety_first_score(plan_score)` function that
recomputes the composite with these weights, and identify which plan wins.

In [ ]:
def safety_first_score(score: PlanScore) -> float:
    # TODO(you): return weighted composite with safety=0.6, feasibility=0.3, efficiency=0.1
    pass


reranked = sorted(
    enumerate(all_scored, 1),
    key=lambda x: safety_first_score(x[1][1]),
    reverse=True
)

print('Plans re-ranked by safety-first scoring:')
for rank, (plan_idx, (plan, score)) in enumerate(reranked, 1):
    sf = safety_first_score(score)
    assert sf is not None, 'safety_first_score must return a float'
    print(f'  Rank {rank}: Plan {plan_idx}  '
          f'(safety={score.safety:.1f}, composite_orig={score.composite:.2f}, '
          f'safety_first={sf:.2f})')

print('passed ✓')

<details><summary>Solution — Exercise A</summary>

```python
class PlanExecutor:
    def __init__(self, tools):
        self.tools = tools

    def run(self, plan):
        for i, step in enumerate(plan):
            result = self.tools[step.action](**step.params)
            step.result = result
            step.executed = True
            if result is None:
                return f'FAILED:{i}'
        return plan[-1].result
```

</details>

<details><summary>Solution — Exercise B</summary>

```python
def safety_first_score(score: PlanScore) -> float:
    return 0.3 * score.feasibility + 0.1 * score.efficiency + 0.6 * score.safety
```

</details>